In [9]:
import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data
import torch.nn.functional as F

# Create a class that performs aggregation of neighbor node embeddings across multiple relation types in the message passing framework for heterogeneous graphs.

In [5]:
class BlockDiagonalRGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels, num_relations, num_blocks):
        super().__init__(aggr='add')
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_relations = num_relations
        self.num_blocks = num_blocks
        
        # Block size for input and output
        self.in_block_size = in_channels // num_blocks
        self.out_block_size = out_channels // num_blocks
        
        # Weight parameters: one set of blocks per relation
        # Shape: (num_relations, num_blocks, out_block_size, in_block_size)
        self.weight = nn.Parameter(
            torch.randn(num_relations, num_blocks, 
                       self.out_block_size, self.in_block_size)
        )
        
    def forward(self, x, edge_index, edge_type):
        """
        x: Node features [num_nodes, in_channels]
        edge_index: Edge indices [2, num_edges]
        edge_type: Edge type for each edge [num_edges]
        """
        # Propagate messages for each relation type separately
        out = torch.zeros(x.size(0), self.out_channels, device=x.device)
        
        for r in range(self.num_relations):
            # Get edges for this relation type
            mask = edge_type == r
            edge_index_r = edge_index[:, mask]
            
            # Propagate with relation-specific transformation
            out += self.propagate(edge_index_r, x=x, relation=r)
            
        return out
    
    def message(self, x_j, relation):
        """
        x_j: Source node features [num_edges, in_channels]
        relation: Scalar indicating which relation type
        """
        # Apply block diagonal transformation
        # Reshape x_j: [num_edges, num_blocks, in_block_size]
        x_j_blocks = x_j.view(-1, self.num_blocks, self.in_block_size)
        
        # Get weight blocks for this relation: [num_blocks, out_block_size, in_block_size]
        W_r = self.weight[relation]
        
        # Batched matrix multiply
        # x_j_blocks: [num_edges, num_blocks, in_block_size, 1]
        # W_r: [num_blocks, out_block_size, in_block_size]
        # Result: [num_edges, num_blocks, out_block_size]
        out_blocks = torch.matmul(W_r, x_j_blocks.unsqueeze(-1)).squeeze(-1)
        
        # Flatten back: [num_edges, out_channels]
        return out_blocks.view(-1, self.out_channels)

Create a test heterogeneous graph.

In [17]:
# Create a small heterogeneous graph
num_nodes = 10
in_channels = 8
num_relations = 3
out_channels = 8
num_blocks = 4

# Node features: 10 nodes, each with 8 features
x = torch.randn(num_nodes, in_channels)

# Edge index: let's create some edges
# Format: [2, num_edges] where row 0 is source, row 1 is target
edge_index = torch.tensor([
    [0, 1, 1, 2, 3, 3, 4, 5, 6, 7, 8, 9],  # source nodes
    [1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9, 0]   # target nodes
], dtype=torch.long)

# Edge types: assign a relation type to each edge
# We have 12 edges, so we need 12 type labels (values 0, 1, or 2 for 3 relation types)
edge_type = torch.tensor([0, 1, 0, 2, 1, 0, 2, 1, 0, 2, 1, 0], dtype=torch.long)

Create an instance of the `BlockDiagnoalRGCNConv` class and pass the test data through to make sure it works.

In [18]:
# Create the layer
layer = BlockDiagonalRGCNConv(
    in_channels=in_channels,      # Input: 8 features per node
    out_channels=out_channels,     # Output: 8 features per node
    num_relations=num_relations,    # 3 different relation types
    num_blocks=num_blocks        # 4 blocks (so each block is 2x2)
)

# Forward pass
out = layer(x, edge_index, edge_type)

print(f"Input shape: {x.shape}")           # torch.Size([10, 8])
print(f"Output shape: {out.shape}")        # torch.Size([10, 8])
print(f"Number of edges: {edge_index.shape[1]}")  # 12
print(f"Edge types: {edge_type}")          # tensor([0, 1, 0, 2, 1, 0, 2, 1, 0, 2, 1, 0])

# You can also check the number of parameters
num_params = layer.weight.numel()
print(f"Number of parameters: {num_params}")  # 3 relations × 4 blocks × 2 × 2 = 48

Input shape: torch.Size([10, 8])
Output shape: torch.Size([10, 8])
Number of edges: 12
Edge types: tensor([0, 1, 0, 2, 1, 0, 2, 1, 0, 2, 1, 0])
Number of parameters: 48


The above only performs the neighbor node aggregation step. We also want to add a self loop for the current node to the aggregated neighbor nodes and pass through a nonlinear activation.

In [24]:
import torch.nn.functional as F
self_weight = nn.Parameter(torch.randn(out_channels, in_channels))
x = torch.randn(num_nodes, in_channels)
x_original = x.clone()  # Save original features for self-loop
# In your model's forward pass
x = layer(x, edge_index, edge_type)
x_original = self_weight @ x_original.t()  # Self-connection
x = x + x_original.t()  # Add self-connection
x = F.relu(x)  # Activation
print(x.shape)

torch.Size([10, 8])


Putting it all together (TODO)


~~~
class BlockDiagonalRGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels, num_relations, num_blocks, 
                 activation=None, add_self_loops=True):
        super().__init__(aggr='add')
        
        # ... existing initialization ...
        
        self.activation = activation
        
        # Optional self-loop weight
        if add_self_loops:
            self.self_weight = nn.Parameter(torch.randn(out_channels, in_channels))
        else:
            self.self_weight = None
            
        self.bias = nn.Parameter(torch.zeros(out_channels))
        
    def forward(self, x, edge_index, edge_type):
        # Neighbor aggregation
        out = # ... existing code ...
        
        # Add self-loops if enabled
        if self.self_weight is not None:
            out = out + x @ self.self_weight.t()
        
        # Add bias
        out = out + self.bias
        
        # Apply activation if specified
        if self.activation is not None:
            out = self.activation(out)
            
        return out)
~~~